In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

In [ ]:
from tempfile import TemporaryDirectory
from klygo import files, utils

if 'tutorial_workspace' in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix='klygo-utils-progress-')
root = files.path(tutorial_workspace.name)

# Progress

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/utils/Document_Utils_Progress.ipynb)


## Jump to an API

- [utils.ProgressBar](#utils.ProgressBar)
- [utils.create_progress_bar](#utils.create_progress_bar)

### utils.ProgressBar

`utils.ProgressBar(total, desc, unit='file', verbose=True, colour='cyan', unit_scale=True, unit_divisor=1000)`

- `total`: expected count; `None` or zero uses a display fallback of one.
- `desc`, `unit`, `colour`: display text and color.
- `verbose`: disable rendering when false.
- `unit_scale`, `unit_divisor`: counter scaling.
- Returns a context-managed progress wrapper.

#### Basic use

In [ ]:
with utils.ProgressBar(3, 'Loading', verbose=False) as progress:
    for _ in range(3):
        progress.update()
print(progress.bar)

#### Practical variation

Count locally created files; use `verbose=False` for quiet scripts.

In [ ]:
with utils.ProgressBar(2, 'Writing', unit='file', verbose=False) as progress:
    for index in range(2):
        files.save(files.join(root, f'{index}.txt'), str(index), verbose=False)
        progress.update()

#### Expected error and recovery

ProgressBar does not validate the meaning of a count. Validate your own input first.

In [ ]:
count = -1
try:
    if count < 0:
        raise ValueError('count must be non-negative')
    with utils.ProgressBar(count, 'Work', verbose=False) as progress:
        progress.update()
except ValueError as error:
    print(error)

### utils.create_progress_bar

`utils.create_progress_bar(total, desc, unit='file', verbose=True, colour='cyan', unit_scale=False, unit_divisor=1024) -> ProgressBar`

- Accepts the same display options as `ProgressBar`.
- Returns a configured `ProgressBar` with scaling off by default.

#### Basic use

In [ ]:
with utils.create_progress_bar(2, 'Checking', verbose=False) as progress:
    progress.update(2)

#### Practical variation

Construct a byte counter with scaling enabled.

In [ ]:
with utils.create_progress_bar(2048, 'Bytes', unit='byte', unit_scale=True, verbose=False) as progress:
    progress.update(2048)

#### Expected error and recovery

If configuration is invalid for the renderer, its exception propagates. A quiet bar avoids creating that renderer.

In [ ]:
with utils.create_progress_bar(1, 'Quiet', verbose=False) as progress:
    progress.update()
print('Renderer:', progress.bar)

## Put it together

Use the factory to count files and the class to report a second pass.

In [ ]:
paths = files.find(root, '*.txt')
with utils.create_progress_bar(len(paths), 'Read', verbose=False) as progress:
    for path in paths:
        print(files.name(path), files.load(path, verbose=False))
        progress.update()
with utils.ProgressBar(1, 'Finished', verbose=False) as progress:
    progress.update()

## Remember

Close bars with `with`; use `verbose=False` when display is unwanted. Return to the [utils overview](Document_Utils_Overview.ipynb), visit [formatting](Document_Utils_Formatting.ipynb), or use progress in [archive](../archive/Document_Archive_Overview.ipynb) and [files](../files/Document_Files_Overview.ipynb).